# Adjudication Report

This notebook provides a detailed view of discrepancies between annotators for archaeological NER.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import argilla as rg
import ipywidgets as widgets
from IPython.display import display, HTML
from archaeo_ner_greek.utils import (
    get_argilla_client, 
    load_credentials_from_env, 
    get_dataset_as_dataframe, 
    prepare_iaa_data, 
    generate_adjudication_report_df, 
    highlight_entities_html, \
    highlight_diff_entities_html
)

# 1. Setup Argilla
env_values = load_credentials_from_env()
client = get_argilla_client(env_values)

# 2. Load Dataset
workspace_name = "atrium"
dataset_name = "atrium_csd_ner_annotations"
df = get_dataset_as_dataframe(client, dataset_name=dataset_name, workspace_name=workspace_name, include_responses=True)

# 3. Prepare IAA Data
iaa_results = prepare_iaa_data(df)
iaa_ready = iaa_results["ready"]

annotator_a = "stalexan"
annotator_b = "tim.evans"

print(f"Loaded {len(iaa_ready)} sentences for adjudication between {annotator_a} and {annotator_b}.")

# 4. Generate the detailed adjudication report (needed for interactive tool)
adj_df = generate_adjudication_report_df(iaa_ready, annotators=[annotator_a, annotator_b])
discrepancies_only = adj_df[adj_df["status"] != "Match"].copy()
print(f"🔍 Total discrepancies to review: {len(discrepancies_only)}")

## Interactive Adjudication Tool

Use the toggle and navigation buttons below to review discrepancies.

In [ ]:
# Filter iaa_ready to match the discrepancies-only index mapping
discrepancy_ids = set(discrepancies_only["sentence_id"])
iaa_discrepancies = [item for item in iaa_ready if item["sentence_id"] in discrepancy_ids]

# Create widgets
record_slider = widgets.IntSlider(
    value=0, min=0, max=len(iaa_discrepancies)-1, 
    description='Sentence:', 
    layout=widgets.Layout(width='40%')
)

prev_button = widgets.Button(description='Previous', icon='chevron-left')
next_button = widgets.Button(description='Next', icon='chevron-right')

annotator_toggle = widgets.Dropdown(
    options=[(annotator_a, annotator_a), (annotator_b, annotator_b), ('Compare (Stacked)', 'compare'), ('Diff (Merge)', 'diff')],
    value=annotator_a,
    description='View:',
)

# Use HTML widget for stability - clears previous content automatically on .value update
html_viewer = widgets.HTML()

def update_view(change):
    if len(iaa_discrepancies) == 0: 
        html_viewer.value = "No discrepancies found."
        return
        
    idx = record_slider.value
    user = annotator_toggle.value
    item = iaa_discrepancies[idx]
    
    text = item["text"]
    
    # Get discrepancy info from our report DF
    report_row = discrepancies_only[discrepancies_only["sentence_id"] == item["sentence_id"]].iloc[0]
    
    html_out = f"<h3>Sentence ID: {item['sentence_id']} ({idx + 1} / {len(iaa_discrepancies)})</h3>"
    
    if user == 'compare':
        spans_a = item["annotations"].get(annotator_a, [])
        spans_b = item["annotations"].get(annotator_b, [])
        
        html_out += f"<div style='border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;'>"
        html_out += f"<h4 style='margin-top: 0;'>{annotator_a}</h4>"
        if annotator_a not in item["annotations"]:
            html_out += f"<p style='color: red;'><b>Warning:</b> User '{annotator_a}' not found.</p>"
        html_out += highlight_entities_html(text, spans_a)
        html_out += "</div>"
        
        html_out += f"<div style='border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;'>"
        html_out += f"<h4 style='margin-top: 0;'>{annotator_b}</h4>"
        if annotator_b not in item["annotations"]:
            html_out += f"<p style='color: red;'><b>Warning:</b> User '{annotator_b}' not found.</p>"
        html_out += highlight_entities_html(text, spans_b)
        html_out += "</div>"
        
    elif user == 'diff':
        spans_a = item["annotations"].get(annotator_a, [])
        spans_b = item["annotations"].get(annotator_b, [])
        html_out += f"<div style='border: 1px solid #ccc; padding: 10px; margin-bottom: 10px;'>"
        html_out += f"<h4 style='margin-top: 0;'>Combined Diff (Showing Differences Only)</h4>"
        
        # We need to make sure highlight_diff_entities_html is imported in the setup cell
        html_out += highlight_diff_entities_html(text, spans_a, spans_b, annotator_a, annotator_b)
        html_out += "</div>"
        
    else:
        spans = item["annotations"].get(user, [])
        html_out += f"<p><b>Showing annotations for:</b> {user}</p>"
        
        if user not in item["annotations"]:
            html_out += f"<p style='color: red;'><b>Warning:</b> User '{user}' not found in record. Available keys: {list(item['annotations'].keys())}</p>"
        
        html_out += highlight_entities_html(text, spans)
        
    html_out += "<hr>"
    html_out += "<h4>Discrepancy Details (Ref):</h4>"
    html_out += f"<pre style='background: #f8f9fa; padding: 10px; border: 1px solid #ddd;'>{report_row['discrepancies']}</pre>"
    
    # Simple assignment ensures NO duplication
    html_viewer.value = html_out

def on_prev_clicked(b):
    if record_slider.value > 0:
        record_slider.value -= 1

def on_next_clicked(b):
    if record_slider.value < record_slider.max:
        record_slider.value += 1

record_slider.observe(update_view, names='value')
annotator_toggle.observe(update_view, names='value')
prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Initial display
nav_box = widgets.HBox([prev_button, record_slider, next_button])
ui = widgets.VBox([nav_box, annotator_toggle])
display(ui)
display(html_viewer)
update_view(None)

## Discrepancy Statistics

In [ ]:
# Summary Statistics for the discrepancies identified above

def style_adjudication(row):
    styles = [''] * len(row)
    if row['status'] == 'Label Mismatch':
        return ['background-color: #fff3cd'] * len(row) # Warning/Yellow
    elif 'Miss' in row['status']:
        return ['background-color: #f8d7da'] * len(row) # Danger/Red
    elif row['status'] == 'Boundary Mismatch':
        return ['background-color: #e2e3e5'] * len(row) # Info/Gray
    elif row['status'] == 'Multiple Issues':
        return ['background-color: #d1ecf1'] * len(row) # Cyan
    return styles

display(discrepancies_only.style
    .apply(style_adjudication, axis=1)
    .set_properties(**{
        'text-align': 'left',
        'white-space': 'pre-wrap',
        'vertical-align': 'top'
    })
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('border', '1px solid #dee2e6')]}
    ])
)